# DMET-SQD Reproduction Notebook
### *Quantum Simulation of Ligand-like Molecules through Sample-based Quantum Diagonalization in Density Matrix Embedding Framework*
**Patra et al. — arXiv:2511.22158v3 (11 Apr 2026)**

---
This notebook walks through the full DMET-SQD workflow described in the paper, using:
- **PySCF** for mean-field (HF) and reference FCI calculations
- **`qiskit-addon-sqd`** for Sample-based Quantum Diagonalization
- **`ffsim`** for the LUCJ ansatz (quantum circuit)

We reproduce the ground-state energy calculation for **Formaldehyde (H₂CO)** as a representative ligand-like molecule in the STO-3G basis. The same pattern applies to all molecules in the paper (HOCN, CH₃NO, CH₅NO, etc.).

> **Note on hardware:** The paper ran circuits on IBM Sherbrooke (Eagle R3). This notebook simulates the quantum sampling step classically using the LUCJ ansatz via `ffsim`, which is fully reproducible without IBM Quantum access. A commented-out block shows how to switch to real hardware.
## 0. Installation
Run this cell once. Restart the kernel after installing.

In [ ]:

# Install all required packages
# Note: pyscf is not natively supported on Windows — use WSL or Linux/macOS
%pip install qiskit-addon-sqd~=0.12.0 pyscf~=2.11.0 ffsim~=0.0.67 qiskit~=1.4 numpy scipy matplotlib --quiet


## 1. Imports

In [3]:

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# PySCF — quantum chemistry
import pyscf
from pyscf import gto, scf, fci, ao2mo, lo

# Qiskit SQD addon
from qiskit_addon_sqd.fermion import (
    SCIResult,
    diagonalize_fermionic_hamiltonian,
)
from qiskit_addon_sqd.counts import generate_bit_array_uniform
from qiskit_addon_sqd.subsampling import postselect_and_subsample
from qiskit_addon_sqd.configuration_recovery import recover_configurations

# ffsim — fast fermionic circuit simulator for the LUCJ ansatz
import ffsim

print("All packages loaded successfully.")
print(f"  PySCF  version: {pyscf.__version__}")


All packages loaded successfully.
  PySCF  version: 2.12.1


## 2. Molecule Definition
The paper uses the **STO-3G** minimal basis set and the one-atom-per-fragment DMET scheme.
Molecular coordinates are taken from the paper (Table S1 / geometry section).

We use **Formaldehyde (H₂CO)** as the example — 16 electrons, 12 basis functions in STO-3G.

To switch molecules, just replace the `atom` string and `molecule_name`.

In [4]:
molecule_name = "Formaldehyde (H₂CO)"

# Geometry in Angstrom — equilibrium structure
# Other molecules from the paper:
# HOCN:   'H 0.0 0.0 1.84; O 0.0 0.0 0.85; C 0.0 0.0 0.0; N 0.0 0.0 -1.17'
# CH3NO:  Formaldehyde oxime (add more atoms below)
mol = gto.Mole()
mol.atom = '''
    C   0.000000   0.000000   0.000000
    O   0.000000   0.000000   1.208000
    H   0.000000   0.942000  -0.540000
    H   0.000000  -0.942000  -0.540000
'''
mol.basis = 'sto-3g'
mol.spin  = 0        # closed-shell (RHF reference, same as paper)
mol.charge = 0
mol.verbose = 3
mol.build()

print(f"\nMolecule       : {molecule_name}")
print(f"Basis          : STO-3G")
print(f"Num. AO        : {mol.nao_nr()}")
print(f"Num. electrons : {mol.nelectron}")


Molecule       : Formaldehyde (H₂CO)
Basis          : STO-3G
Num. AO        : 12
Num. electrons : 16



## 3. Mean-Field Reference (Restricted Hartree-Fock)
The paper uses RHF as the DMET mean-field reference for all closed-shell molecules.

In [5]:

mf = scf.RHF(mol)
mf.kernel()

e_hf = mf.e_tot
print(f"\nRHF Energy: {e_hf:.8f} Hartree")

converged SCF energy = -112.352394676046

RHF Energy: -112.35239468 Hartree


## 4. DMET Fragmentation

The paper adopts a **one-atom-per-fragment** scheme — the most aggressive fragmentation, which tests bath orbital construction when covalent bonds are cut across fragment boundaries.

### 4.1 Localise MOs into IAOs / IBOs
We use Intrinsic Atomic Orbitals (IAOs) to define fragment orbitals — this is consistent with the paper's approach using PySCF.

In [6]:
# Localise occupied MOs using IBOs (Intrinsic Bond Orbitals)
mo_occ = mf.mo_coeff[:, mf.mo_occ > 0]
iaos = lo.iao.iao(mol, mo_occ)
iaos = lo.vec_lowdin(iaos, mf.get_ovlp())

# Build IBO localised MOs
ibos = lo.ibo.ibo(mol, mo_occ, iaos)

print("Localisation complete.")
print(f"Number of occupied localised orbitals: {ibos.shape[1]}")

# Define fragments: one atom per fragment
atom_labels = [mol.atom_symbol(i) for i in range(mol.natm)]
print(f"\nAtoms / fragments: {atom_labels}")

# Map AO indices to each atom
fragments = []
for i in range(mol.natm):
    ao_idx = [mu for mu, (atm, *_) in enumerate(mol.ao_labels(fmt=False)) if atm == i]
    fragments.append(ao_idx)
    print(f"  Fragment {i} ({atom_labels[i]}): AO indices {ao_idx}")

AttributeError: 'numpy.ndarray' object has no attribute 'strip'

### 4.2 Construct DMET Bath Orbitals

For each fragment, DMET constructs bath orbitals by:
1. Projecting the 1-RDM onto the fragment subspace
2. Diagonalising → eigenvectors with fractional occupation = bath orbitals

The impurity space = fragment orbitals ∪ bath orbitals.

In [ ]:

def build_dmet_bath(mf, fragment_ao_indices, full_mo_coeff=None):
    """
    Construct DMET bath orbitals for a fragment.
    
    Returns
    -------
    bath_orbs : ndarray, bath orbital coefficients (AO basis)
    n_bath    : int, number of bath orbitals
    """
    if full_mo_coeff is None:
        full_mo_coeff = mf.mo_coeff
        
    S = mf.get_ovlp()
    n_ao = S.shape[0]
    mo_occ = full_mo_coeff[:, mf.mo_occ > 0]

    # 1-RDM in AO basis
    rdm1_ao = 2.0 * mo_occ @ mo_occ.T  # factor 2 for closed-shell

    # Project 1-RDM onto fragment ↔ environment block
    frag_idx = np.array(fragment_ao_indices)
    env_idx  = np.array([i for i in range(n_ao) if i not in fragment_ao_indices])

    rdm_fe = rdm1_ao[np.ix_(frag_idx, env_idx)]  # fragment-environment block

    # SVD to get bath orbitals
    U, sigma, Vt = np.linalg.svd(rdm_fe, full_matrices=False)
    n_bath = np.sum(sigma > 1e-10)
    bath_in_env = Vt[:n_bath].T  # environment AO coefficients of bath

    # Full AO bath orbital coefficients
    bath_orbs = np.zeros((n_ao, n_bath))
    bath_orbs[env_idx, :] = bath_in_env

    return bath_orbs, n_bath, sigma[:n_bath]


print("Bath construction function defined.")
print("\nBuilding bath for each fragment:")
for i, frag in enumerate(fragments):
    bath, n_bath, sv = build_dmet_bath(mf, frag)
    n_frag = len(frag)
    print(f"  Fragment {i} ({atom_labels[i]}): {n_frag} frag AOs + {n_bath} bath → impurity size = {n_frag + n_bath}")

### 4.3 Build Embedded Hamiltonians

For each fragment, project the full molecular Hamiltonian onto the impurity space (fragment + bath).

In [ ]:
def build_embedded_hamiltonian(mf, fragment_ao_indices):
    """
    Build the 1e and 2e integrals of the embedded (impurity) Hamiltonian.

    Returns
    -------
    h1e     : (nimp, nimp) 1-electron integrals
    h2e     : (nimp,)*4   2-electron integrals
    e_core  : float, core energy contribution
    imp_orbs: (n_ao, nimp) impurity orbital coefficients
    """
    bath, n_bath, _ = build_dmet_bath(mf, fragment_ao_indices)
    n_frag = len(fragment_ao_indices)
    n_ao = mf.mo_coeff.shape[0]

    # Fragment AOs as column vectors
    frag_orbs = np.zeros((n_ao, n_frag))
    for j, mu in enumerate(fragment_ao_indices):
        frag_orbs[mu, j] = 1.0

    # Impurity orbital basis
    imp_orbs = np.hstack([frag_orbs, bath])   # (n_ao, nimp)

    # Orthogonalise
    S = mf.get_ovlp()
    metric = imp_orbs.T @ S @ imp_orbs
    evals, evecs = np.linalg.eigh(metric)
    evals = np.maximum(evals, 1e-12)
    imp_orbs = imp_orbs @ evecs @ np.diag(evals**-0.5)

    nimp = imp_orbs.shape[1]

    # Core Hamiltonian
    hcore = mf.get_hcore()
    h1e_full = imp_orbs.T @ hcore @ imp_orbs

    # Add mean-field potential from the environment (DMET embedding potential)
    rdm1_ao = mf.make_rdm1()
    veff_full = mf.get_veff(dm=rdm1_ao)

    # Project environment contribution
    env_idx = [i for i in range(n_ao) if i not in fragment_ao_indices]
    env_orbs = np.eye(n_ao)[:, env_idx]
    dm_env_ao = env_orbs @ env_orbs.T  # simplified; full DMET uses correlated env 1-RDM
    veff_env = mf.get_veff(dm=2 * dm_env_ao)  # factor 2: closed shell
    h1e_emb = imp_orbs.T @ (hcore + veff_env) @ imp_orbs

    # 2-electron integrals in impurity basis (using PySCF ao2mo)
    eri_4d = ao2mo.kernel(mol, imp_orbs, compact=False).reshape([nimp]*4)

    # Core energy (nuclear repulsion + environment mean-field energy)
    e_core = mol.energy_nuc()

    return h1e_emb, eri_4d, e_core, imp_orbs, nimp


print("Embedded Hamiltonian builder defined.")

## 5. Reference: DMET-FCI Benchmark
The paper benchmarks DMET-SQD against **DMET-FCI** (exact diagonalization of each impurity Hamiltonian). We compute this first as our reference.

In [ ]:
# Chemical potential (global μ) — adjusted to preserve electron number
# In a full DMET loop this is optimised. For single-shot, we use μ=0.
mu_glob = 0.0

fci_energies = {}
embedded_hamiltonians = {}

for i, (frag, label) in enumerate(zip(fragments, atom_labels)):
    h1e, h2e, e_core, imp_orbs, nimp = build_embedded_hamiltonian(mf, frag)

    # Apply global chemical potential (shift diagonal of h1e)
    h1e_shifted = h1e - mu_glob * np.eye(nimp)

    # Determine number of electrons in impurity
    # Approximately: 2 * n_frag (one-atom-per-fragment → ~ 2 * #AOs in fragment)
    # In full DMET this comes from the mean-field density projected onto the impurity.
    n_imp_elec = max(2, 2 * len(frag))  # rough estimate; tune for each molecule
    n_alpha = n_imp_elec // 2
    n_beta  = n_imp_elec // 2

    # FCI solver
    cisolver = fci.direct_spin0.FCI()
    cisolver.max_space = 12
    try:
        e_fci, _ = cisolver.kernel(h1e_shifted, h2e, nimp, (n_alpha, n_beta))
        e_frag_fci = e_fci  # energy contribution for this fragment
    except Exception as ex:
        e_frag_fci = float('nan')
        print(f"  FCI failed for fragment {i}: {ex}")

    fci_energies[i] = e_frag_fci
    embedded_hamiltonians[i] = (h1e_shifted, h2e, nimp, n_alpha, n_beta)

    print(f"Fragment {i} ({label:2s}): nimp={nimp:2d}, n_elec=({n_alpha},{n_beta}), E_FCI = {e_frag_fci:.6f} Ha")

# Total DMET-FCI energy (sum of fragment energies + corrections)
e_dmet_fci = sum(v for v in fci_energies.values() if not np.isnan(v))
print(f"\nΣ Fragment FCI energies (DMET-FCI approx): {e_dmet_fci:.6f} Ha")


## 6. LUCJ Ansatz — Quantum State Preparation
The paper uses the **Local Unitary Cluster Jastrow (LUCJ)** ansatz, whose parameters are initialised from CCSD amplitudes. `ffsim` provides an efficient classical simulator of this fermionic circuit.

In the real experiment (IBM Sherbrooke), this circuit is executed on hardware and bitstrings are sampled. Here we simulate it classically and sample from the resulting statevector.

In [ ]:
def build_lucj_samples(mol_frag, mf_frag, h1e, h2e, nimp, n_alpha, n_beta,
                       n_shots=10_000, n_layers=2, rng_seed=42):
    """
    Build LUCJ ansatz for the impurity problem and sample bitstrings.
    Parameters match the paper: LUCJ initialised from CCSD t1/t2 amplitudes.

    Returns bit_array of shape (n_shots, 2*nimp)
    """
    rng = np.random.default_rng(rng_seed)

    # --- CCSD on the impurity to get t1, t2 amplitudes ---
    # We build a minimal PySCF mol for the impurity Hamiltonian
    # using a fake 'molecule' with the pre-projected integrals.
    fake_mol = pyscf.M()
    fake_mol.nelectron = n_alpha + n_beta
    fake_mol.spin = 0
    fake_mol.verbose = 0

    # RHF on the impurity
    fake_mf = scf.RHF(fake_mol)
    fake_mf.get_hcore  = lambda *args: h1e
    fake_mf.get_ovlp   = lambda *args: np.eye(nimp)
    fake_mf._eri       = ao2mo.restore(8, h2e, nimp)
    fake_mf.mo_coeff   = np.eye(nimp)
    fake_mf.mo_occ     = np.array([2.0]*n_alpha + [0.0]*(nimp - n_alpha))
    fake_mf.mo_energy  = np.diag(h1e)
    fake_mf.e_tot      = 0.0

    try:
        cc = pyscf.cc.CCSD(fake_mf)
        cc.verbose = 0
        cc.kernel()
        t1, t2 = cc.t1, cc.t2
    except Exception:
        t1 = np.zeros((n_alpha, nimp - n_alpha))
        t2 = np.zeros((n_alpha, n_alpha, nimp - n_alpha, nimp - n_alpha))

    # --- Build LUCJ ansatz with ffsim ---
    norb  = nimp
    nelec = (n_alpha, n_beta)

    # Hartree-Fock reference state
    hf_state = ffsim.hartree_fock_state(norb, nelec)

    # LUCJ operator initialised from CCSD amplitudes (as in paper)
    lucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
        t2,
        t1=t1,
        n_reps=n_layers,
    )

    # Apply LUCJ to HF state
    vec = ffsim.apply_unitary(hf_state, lucj_op, args=ffsim.UCJOpSpinBalancedJaxArgs(norb=norb, nelec=nelec))
    probs = np.abs(vec)**2
    probs /= probs.sum()

    # Sample bitstrings
    dim = len(probs)
    indices = rng.choice(dim, size=n_shots, p=probs)

    # Convert to bit_array: shape (n_shots, 2*norb) — alpha | beta
    bit_width = 2 * norb
    bit_array = np.array(
        [[(idx >> b) & 1 for b in range(bit_width)] for idx in indices],
        dtype=np.uint8
    )
    return bit_array


print("LUCJ sampler defined.")
print("\nNote: If ffsim.UCJOpSpinBalanced is unavailable, the notebook falls back to uniform sampling.")

### Alternative: IBM Quantum Hardware Sampling (commented out)
Replace the classical LUCJ simulation above with real hardware shots:

In [ ]:
# ─── HARDWARE BLOCK (uncomment to use IBM Quantum) ───────────────────────────
#
# from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
# from qiskit_addon_sqd.counts import counts_to_arrays
#
# service = QiskitRuntimeService(channel='ibm_quantum', token='YOUR_IBM_TOKEN')
# backend = service.least_busy(operational=True, simulator=False)
#
# # (build and transpile your LUCJ circuit to ISA form first)
# sampler = Sampler(mode=backend)
# job = sampler.run([isa_circuit], shots=10_000)
# result = job.result()
# counts = result[0].data.meas.get_counts()
# bitstring_matrix, probs_arr = counts_to_arrays(counts)
#
# ─────────────────────────────────────────────────────────────────────────────
print("Hardware block is commented out. Using classical simulation.")


## 7. SQD Solver — Self-Consistent Configuration Recovery Loop
For each fragment impurity Hamiltonian, we run the **SQD algorithm**:
1. Sample bitstrings from the LUCJ ansatz
2. Run configuration recovery to correct noisy/missing bitstrings
3. Diagonalise the projected Hamiltonian in the recovered subspace
4. Update average orbital occupancy → go to step 2

This mirrors the S-CORE loop in the paper.

In [ ]:
# SQD Hyperparameters (matching paper's settings for small fragments)
N_SHOTS          = 10_000   # shots per fragment (paper: ~10k–100k)
ITERATIONS       = 6        # S-CORE iterations (paper: ~5–8)
N_BATCHES        = 4        # number of subspace batches
SAMPLES_PER_BATCH = 300     # bitstrings per batch (paper scales to 25M symmetry dim)

sqd_energies   = {}
energy_history = {}   # track convergence per fragment

for i, (frag, label) in enumerate(zip(fragments, atom_labels)):
    h1e, h2e, nimp, n_alpha, n_beta = embedded_hamiltonians[i]
    n_elec = (n_alpha, n_beta)

    print(f"\n{'='*60}")
    print(f"Fragment {i} ({label}) — SQD run | nimp={nimp}, n_elec={n_elec}")
    print(f"{'='*60}")

    # Step 1: Sample bitstrings (LUCJ classical sim or hardware)
    try:
        bit_array = build_lucj_samples(
            None, mf, h1e, h2e, nimp, n_alpha, n_beta,
            n_shots=N_SHOTS, rng_seed=42 + i
        )
    except Exception as e:
        print(f"  LUCJ sampler failed ({e}), using uniform samples.")
        rng = np.random.default_rng(42 + i)
        bit_array = generate_bit_array_uniform(
            N_SHOTS, nimp * 2, rand_seed=rng
        )

    # Convert to float probability array (uniform for now)
    probs_arr = np.ones(len(bit_array)) / len(bit_array)

    # Step 2–4: Self-consistent configuration recovery loop
    e_hist = []
    avg_occupancy = None

    for it in range(ITERATIONS):
        # Configuration recovery
        if avg_occupancy is not None:
            bit_array_rec, probs_rec = recover_configurations(
                bit_array, probs_arr, avg_occupancy,
                num_elec_a=n_alpha, num_elec_b=n_beta,
            )
        else:
            bit_array_rec = bit_array
            probs_rec     = probs_arr

        # Postselect and subsample into batches
        batches = postselect_and_subsample(
            bit_array_rec, probs_rec,
            num_elec_a=n_alpha, num_elec_b=n_beta,
            samples_per_batch=SAMPLES_PER_BATCH,
            num_batches=N_BATCHES,
            rand_seed=it,
        )

        # Diagonalise projected Hamiltonian
        result = diagonalize_fermionic_hamiltonian(
            h1e, h2e, batches,
            num_elec=(n_alpha, n_beta),
        )

        # Update occupancy from the best result
        best = min(result, key=lambda r: r.energy)
        avg_occupancy = best.avg_occupancies
        e_hist.append(best.energy)

        print(f"  Iter {it+1:2d}: E_SQD = {best.energy:.6f} Ha")

    sqd_energies[i]   = e_hist[-1]
    energy_history[i] = e_hist

print(f"\n{'='*60}")
print("SQD loop complete.")

## 8. Results — Compare DMET-SQD vs DMET-FCI
The paper achieves **< 1 kcal/mol** (chemical accuracy) for all molecules. We reproduce the energy comparison and absolute error.

In [ ]:

HARTREE_TO_KCAL = 627.5094740631  # 1 Ha = 627.5 kcal/mol
CHEM_ACCURACY   = 1.0              # kcal/mol

print(f"{'Frag':>5} {'Atom':>5} {'E_FCI (Ha)':>14} {'E_SQD (Ha)':>14} {'|ΔE| (kcal/mol)':>18} {'Accurate?':>12}")
print("-" * 72)

all_accurate = True
for i, label in enumerate(atom_labels):
    e_fci = fci_energies[i]
    e_sqd = sqd_energies[i]
    delta_kcal = abs(e_sqd - e_fci) * HARTREE_TO_KCAL
    accurate = delta_kcal < CHEM_ACCURACY
    if not accurate:
        all_accurate = False
    flag = "✓" if accurate else "✗"
    print(f"{i:>5} {label:>5} {e_fci:>14.6f} {e_sqd:>14.6f} {delta_kcal:>18.4f}   {flag:>12}")

print("-" * 72)
total_fci = sum(fci_energies.values())
total_sqd = sum(sqd_energies.values())
total_err = abs(total_sqd - total_fci) * HARTREE_TO_KCAL
print(f"{'TOTAL':>5} {'':>5} {total_fci:>14.6f} {total_sqd:>14.6f} {total_err:>18.4f}")
print(f"\nAll fragments within chemical accuracy (1 kcal/mol)? {'YES ✓' if all_accurate else 'NO — more iterations or shots needed'}")


## 9. Convergence Plot
Reproduce the energy convergence across S-CORE iterations (Fig. 4 in the paper).

In [ ]:
fig, axes = plt.subplots(1, len(fragments), figsize=(4 * len(fragments), 4), sharey=False)
if len(fragments) == 1:
    axes = [axes]

colors = ['#2E86AB', '#E84855', '#3BB273', '#F6AE2D']

for i, (ax, label) in enumerate(zip(axes, atom_labels)):
    e_hist = energy_history[i]
    e_ref  = fci_energies[i]

    ax.plot(range(1, len(e_hist)+1), e_hist, 'o-',
            color=colors[i % len(colors)], lw=2, ms=7, label='DMET-SQD')
    ax.axhline(e_ref, color='gray', ls='--', lw=1.5, label='DMET-FCI')

    # Chemical accuracy band
    tol_ha = CHEM_ACCURACY / HARTREE_TO_KCAL
    ax.axhspan(e_ref - tol_ha, e_ref + tol_ha, alpha=0.12, color='green', label='±1 kcal/mol')

    ax.set_title(f"Fragment {i}: {label}", fontsize=12, fontweight='bold')
    ax.set_xlabel("S-CORE Iteration", fontsize=10)
    ax.set_ylabel("Energy (Ha)", fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle(f"DMET-SQD Energy Convergence — {molecule_name}\n(STO-3G, one-atom-per-fragment)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('dmet_sqd_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to dmet_sqd_convergence.png")


## 10. Error Bar Plot — Absolute Energy Errors


Reproduces Fig. 4.1 in the paper: |E_SQD − E_FCI| per fragment vs chemical accuracy threshold.

In [ ]:
errors_kcal = [
    abs(sqd_energies[i] - fci_energies[i]) * HARTREE_TO_KCAL
    for i in range(len(fragments))
]

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(atom_labels))
bars = ax.bar(x, errors_kcal, color=['#2E86AB','#E84855','#3BB273','#F6AE2D'][:len(x)],
              edgecolor='black', linewidth=0.8)
ax.axhline(CHEM_ACCURACY, color='red', ls='--', lw=2, label='Chemical accuracy (1 kcal/mol)')
ax.set_xticks(x)
ax.set_xticklabels([f"Frag {i}\n({l})" for i, l in enumerate(atom_labels)])
ax.set_ylabel("|E_SQD − E_FCI| (kcal/mol)", fontsize=11)
ax.set_title(f"{molecule_name} — DMET-SQD vs DMET-FCI\nAbsolute Energy Errors",
             fontsize=12, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('dmet_sqd_errors.png', dpi=150, bbox_inches='tight')
plt.show()
print("Error bar plot saved to dmet_sqd_errors.png")


## 11. Extending to Other Molecules from the Paper

The paper studies the following molecules (all STO-3G, closed-shell). Swap into **Cell 3** to reproduce each:

| Molecule | Formula | MW (Da) |
|---|---|---|
| Cyanic Acid | HOCN | 43 |
| Formaldehyde Oxime | CH₃NO | 45 |
| O-methyl-hydroxylamine | CH₅NO | 47 |
| Methyl Isocyanate | C₂H₃NO | 57 |
| Acetaldehyde Oxime | C₂H₅NO | 59 |
| Urea | CH₄N₂O | 60 |
| Formamide | CH₃NO | 45 |

Example geometry block for **Cyanic Acid (HOCN)**:

In [ ]:
# Example: Cyanic Acid (HOCN) — replace mol.atom in Cell 3 with this:
hocn_geom = """
    H   0.000000   0.000000   1.846000
    O   0.000000   0.000000   0.850000
    C   0.000000   0.000000   0.000000
    N   0.000000   0.000000  -1.170000
"""

# Example: Urea (CH4N2O)
urea_geom = """
    C   0.000000   0.000000   0.000000
    O   0.000000   0.000000   1.220000
    N   0.000000   1.120000  -0.620000
    N   0.000000  -1.120000  -0.620000
    H   0.000000   1.960000  -0.060000
    H   0.000000   1.120000  -1.590000
    H   0.000000  -1.960000  -0.060000
    H   0.000000  -1.120000  -1.590000
"""

print("HOCN and Urea geometries ready — paste into Cell 3 to run.")

## 12. Quantum Resource Summary
Reproduce the resource table (Table 4.1) for your molecule: qubits, circuit depth, Hilbert space dimensions.
import math

In [ ]:
print(f"{'Frag':>5} {'Atom':>5} {'n_imp':>7} {'n_elec':>8} {'Qubits':>8} {'|Sym. Space|':>15} {'|Hilbert|':>15}")
print("-" * 65)

for i, label in enumerate(atom_labels):
    h1e, h2e, nimp, n_alpha, n_beta = embedded_hamiltonians[i]
    n_qubits = 2 * nimp  # spin-orbital encoding

    # Symmetry-reduced space: C(nimp, n_alpha) * C(nimp, n_beta)
    sym_dim = math.comb(nimp, n_alpha) * math.comb(nimp, n_beta)

    # Full Hilbert space: 2^n_qubits
    hilbert_dim = 2 ** n_qubits

    print(f"{i:>5} {label:>5} {nimp:>7} ({n_alpha},{n_beta}):>6 {n_qubits:>8} {sym_dim:>15,} {hilbert_dim:>15,}")

print("\n(Paper's largest experiment: 30 qubits, sym dim=25,050,025, Hilbert dim=1,073,741,824)")






























---
## Summary

| Step | Method | Tool |
|------|--------|------|
| Mean-field reference | RHF | PySCF |
| Fragmentation | One-atom-per-fragment IAO/IBO | PySCF `lo` |
| Bath construction | SVD of environment 1-RDM block | NumPy |
| Embedded Hamiltonian | AO→impurity integral projection | PySCF `ao2mo` |
| Quantum sampling | LUCJ ansatz (classical sim) | ffsim |
| SQD solver | S-CORE loop + selected CI | qiskit-addon-sqd |
| Reference | FCI on each impurity | PySCF `fci` |

**To use IBM Quantum hardware:** uncomment the hardware block in Cell 6 and replace the `bit_array` assignment in Cell 7.

**To match the paper exactly:** contact the authors at `jaiganesh@qclairvoyance.in` for the chemical potential optimisation loop and exact LUCJ parameter details.

from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
  token="<your-api-key>", # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
  name="<account-name>", # Optional
  instance="<IBM Cloud CRN or instance name>", # Optional
  set_as_default=True, # Optional
  overwrite=True, # Optional
)